# Principal Component Analysis
1. Standardize data
2. Perform PCA

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
notebook_dir = Path().resolve()
sys.path.append(str(notebook_dir.parent))        # ../
sys.path.append(str(notebook_dir.parent.parent)) # ../../


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
from datetime import datetime

from plot_helper.colors import COLORS

feature_display_names = {
    "hr": "HR [bpm]",
    "rmssd": "HRV [ms]",
    "hr_nocturnal": "Nocturnal HR [bpm]",
    "rmssd_nocturnal": "Nocturnal HRV [ms]",
}



def get_stat_columns(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Return a DataFrame with only columns that start with `prefix`."""
    cols = [c for c in df.columns if c.startswith(prefix)]
    cols.append('study_id')
    if not cols:
        raise ValueError(f"No columns found starting with '{prefix}'. "
                         f"Check STAT_PREFIXES in the config section.")
    sub = df[cols].copy()
    # Strip the prefix from column names so the heatmap shows feature names only
    sub.columns = [c if c=='study_id' else c[len(prefix):] for c in sub.columns]
    return sub

## Load Data

In [ ]:
df_hr = pd.read_csv("../../output/1_feature_extraction/df_features_hr_2026-07-08.csv")
df_nocturnal_hr = pd.read_csv("../../output/1_feature_extraction/df_features_nocturnal_hr_2026-07-08.csv")
#!drop rows where n_nights < 7
df_nocturnal_hr = df_nocturnal_hr[df_nocturnal_hr['n_nights'] >= 7]
df_hr = df_hr[df_hr['n_days'] >= 7]


#keep only columns needed in hr and hr_nocturnal
df_hr = df_hr[['study_id', 'mean_hr', 'mean_rmssd']]
df_nocturnal_hr = df_nocturnal_hr[['study_id', 'mean_hr', 'mean_rmssd']]

df_hr = get_stat_columns(df_hr, "mean_")
df_nocturnal_hr = get_stat_columns(df_nocturnal_hr, "mean_")
df = pd.merge(df_hr, df_nocturnal_hr, on='study_id', how='outer', suffixes=('', '_nocturnal'))


display(df.columns)
#move study_id to first column
cols = df.columns.tolist()
cols.insert(0, cols.pop(cols.index('study_id')))
df = df[cols]
display(df.head())
#store prepared feature table for prediction modelling later
date = datetime.now().strftime("%Y-%m-%d")
df.to_csv(f"../../output/4_Prediction_Model/0_prep_data/df_features_all_hr_prep_{date}.csv", index=False)

# get number of features (-1 because of study_id column) and participants
num_features = len(df.columns)-1
num_participants = len(df)

print(f"Number of features: {num_features}, Number of participants: {num_participants}")

print(df.columns)

feature_cols = [col for col in df.columns if col != 'study_id']
display(feature_cols)

## Standardize & PCA

In [ ]:
# ── 1. LOAD YOUR FINAL FEATURE TABLE ──────────────────────────────────────────
# One row per participant, columns are your selected features
# Replace with your actual file path and feature list
df = df.copy()
df = df.dropna()
display(df.shape)


#feature selection
X = df[feature_cols].copy()
display(X.head())

# ── 2. HANDLE MISSING VALUES ───────────────────────────────────────────────────
# Check how many missing values you have per feature
print("Missing values per feature:")
print(X.isnull().sum())


X = X.dropna()


# ── 3. STANDARDISE (Z-SCORE) ──────────────────────────────────────────────────
# Essential before PCA — brings all features to same scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X[feature_cols]),
    columns=feature_cols,
    index=df.index
)

# ── 4. RUN PCA ────────────────────────────────────────────────────────────────
# Start with n_components = number of features to see full variance picture
pca = PCA(n_components=len(feature_cols))
pca.fit(X_scaled)
scores = pca.transform(X_scaled)  # participant coordinates in PC space

### Scree plot

In [ ]:
# ── 5. SCREE PLOT ─────────────────────────────────────────────────────────────
# Tells you how many PCs are worth keeping
explained_var = pca.explained_variance_ratio_ * 100
cumulative_var = np.cumsum(explained_var)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(range(1, len(explained_var) + 1), explained_var,
        color=COLORS.get("blue1"), alpha=0.7, label="Individual explained variance")
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance (%)", color=COLORS.get("blue1"))
ax1.tick_params(axis="y", labelcolor=COLORS.get("blue1"))

ax2 = ax1.twinx()
ax2.plot(range(1, len(cumulative_var) + 1), cumulative_var,
         color=COLORS.get("red1"), marker="o", label="Cumulative explained variance")
ax2.axhline(y=70, color="gray", linestyle="--", alpha=0.5, label="70% threshold")
ax2.axhline(y=80, color="black", linestyle="--", alpha=0.5, label="80% threshold")
ax2.set_ylabel("Cumulative Variance (%)", color=COLORS.get("red1"))
ax2.tick_params(axis="y", labelcolor=COLORS.get("red1"))        

plt.title("Scree Plot — PCA Explained Variance")
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.85))
plt.tight_layout()
date = datetime.now().strftime("%Y-%m-%d")
plt.savefig(f"../../plots/Prediction_Model/PCA/hr_scree_plot_initial_{date}.png", dpi=300)
plt.show()

# Print variance table
print("\nVariance explained per component:")
for i, (indiv, cumul) in enumerate(zip(explained_var, cumulative_var)):
    print(f"  PC{i+1}: {indiv:.1f}%  (cumulative: {cumul:.1f}%)")
#store variance table in dataframe
variance_table = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(explained_var))],
    'Explained Variance (%)': explained_var,
    'Cumulative Variance (%)': cumulative_var
})
variance_table.to_csv(f"../../output/4_Prediction_Model/PCA/hr_variance_table_{date}.csv", index=False)



### Number of components & Heatmap

In [ ]:
# ── 6. CHOOSE NUMBER OF COMPONENTS ────────────────────────────────────────────
# Rule: keep enough PCs to explain 70–80% of variance
# Look at scree plot for the "elbow" — the point where adding more PCs
# gives diminishing returns
# Set n_components based on what you see
n_components = 2
pca_final = PCA(n_components=n_components)
scores_final = pca_final.fit_transform(X_scaled)

# ── 7. LOADINGS HEATMAP ───────────────────────────────────────────────────────
# Shows how much each original feature contributes to each PC
# This tells you what each PC biologically represents

loadings = pd.DataFrame(
    pca_final.components_.T,
    index=feature_cols,
    columns=[f"PC{i+1}" for i in range(n_components)]
)
loadings_display = loadings.rename(index=lambda f: feature_display_names.get(f, f))

plt.figure(figsize=(8, 10))
sns.heatmap(loadings_display, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"label": "Loading"})
plt.title("PCA Loadings Heatmap")
plt.tight_layout()
date = datetime.now().strftime("%Y-%m-%d")
plt.savefig(f"../../plots/Prediction_Model/PCA/hr_pca_loadings_initial_{date}.png", dpi=300)
plt.show()


### Plot PCs

In [ ]:
# ── 8. PC1 vs PC2 SCATTER PLOT ────────────────────────────────────────────────
# Visual inspection — do natural groups emerge?
plt.figure(figsize=(8, 6))
plt.scatter(scores_final[:, 0], scores_final[:, 1],
            s=80, alpha=0.8, edgecolors="black", linewidths=0.5, color=COLORS.get("blue1"))

# Label each point with participant ID if available
if "study_id" in df.columns:
    for i, pid in enumerate(df["study_id"]):
        plt.annotate(pid,
                     (scores_final[i, 0], scores_final[i, 1]),
                     fontsize=7, alpha=0.7,
                     xytext=(4, 4), textcoords="offset points")

plt.xlabel(f"PC1 ({explained_var[0]:.1f}% variance)")
plt.ylabel(f"PC2 ({explained_var[1]:.1f}% variance)")
plt.title("PCA Score Plot — PC1 vs PC2")
plt.axhline(0, color="gray", linestyle="--", alpha=0.4)
plt.axvline(0, color="gray", linestyle="--", alpha=0.4)
plt.tight_layout()
date = datetime.now().strftime("%Y-%m-%d")
plt.savefig(f"../../plots/Prediction_Model/PCA/hr_pca_scatter_initial_{date}.png", dpi=300)
plt.show()

## Drop Outliers

In [ ]:
# ── 1. LOAD YOUR FINAL FEATURE TABLE ──────────────────────────────────────────
# One row per participant, columns are your selected features
# Replace with your actual file path and feature list
df = df.copy()
df = df.dropna()
display(df.shape)

#!drop outliers
df = df[(df['study_id'].isin(["DEC_59", "DEC_64", "DEC_56"]) == False)]

# Define your final selected features 
features = [col for col in df.columns if col != 'study_id']
X = df[features].copy()

# ── 2. HANDLE MISSING VALUES ───────────────────────────────────────────────────
# Check how many missing values you have per feature
# print("Missing values per feature:")
# print(X.isnull().sum())

X = X.dropna()



# ── 3. STANDARDISE (Z-SCORE) ──────────────────────────────────────────────────
# Essential before PCA — brings all features to same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

# ── 4. RUN PCA ────────────────────────────────────────────────────────────────
# Start with n_components = number of features to see full variance picture
pca = PCA(n_components=len(features))
pca.fit(X_scaled)
scores = pca.transform(X_scaled)  # participant coordinates in PC space

# ── 5. SCREE PLOT ─────────────────────────────────────────────────────────────
# Tells you how many PCs are worth keeping
explained_var = pca.explained_variance_ratio_ * 100
cumulative_var = np.cumsum(explained_var)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(range(1, len(explained_var) + 1), explained_var,
        color=COLORS.get("blue1"), alpha=0.7, label="Individual explained variance")
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance (%)", color=COLORS.get("blue1"))
ax1.tick_params(axis="y", labelcolor=COLORS.get("blue1"))

ax2 = ax1.twinx()
ax2.plot(range(1, len(cumulative_var) + 1), cumulative_var,
         color=COLORS.get("red1"), marker="o", label="Cumulative explained variance")
ax2.axhline(y=70, color="gray", linestyle="--", alpha=0.5, label="70% threshold")
ax2.axhline(y=80, color="black", linestyle="--", alpha=0.5, label="80% threshold")
ax2.set_ylabel("Cumulative Variance (%)", color=COLORS.get("red1"))
ax2.tick_params(axis="y", labelcolor=COLORS.get("red1"))

plt.title("Scree Plot — PCA Explained Variance")
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.85))
plt.tight_layout()
date = datetime.now().strftime("%Y-%m-%d")
plt.savefig(f"../../plots/Prediction_Model/PCA/hr_scree_plot_filtered_{date}.png", dpi=300)
plt.show()

# Print variance table
print("\nVariance explained per component:")
for i, (indiv, cumul) in enumerate(zip(explained_var, cumulative_var)):
    print(f"  PC{i+1}: {indiv:.1f}%  (cumulative: {cumul:.1f}%)")

#store variance table in dataframe
variance_table = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(explained_var))],
    'Explained Variance (%)': explained_var,
    'Cumulative Variance (%)': cumulative_var
})
variance_table.to_csv(f"../../output/4_Prediction_Model/PCA/hr_variance_table_filtered_{date}.csv", index=False)

In [ ]:
# ── 6. CHOOSE NUMBER OF COMPONENTS ────────────────────────────────────────────
# Rule: keep enough PCs to explain 70–80% of variance
# Look at scree plot for the "elbow" — the point where adding more PCs
# gives diminishing returns
# Set n_components based on what you see
n_components = 2
pca_final = PCA(n_components=n_components)
scores_final = pca_final.fit_transform(X_scaled)

# ── 7. LOADINGS HEATMAP ───────────────────────────────────────────────────────
# Shows how much each original feature contributes to each PC
# This tells you what each PC biologically represents

loadings = pd.DataFrame(
    pca_final.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(n_components)]
)
loadings_display = loadings.rename(index=lambda f: feature_display_names.get(f, f))

plt.figure(figsize=(10, 12))
sns.heatmap(loadings_display, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"label": "Loading"})
plt.title("PCA Loadings Heatmap")
plt.tight_layout()
date = datetime.now().strftime("%Y-%m-%d")
plt.savefig(f"../../plots/Prediction_Model/PCA/hr_pca_loadings_filtered_{date}.png", dpi=300)
plt.show()



In [ ]:
# ── 8. PC1 vs PC2 SCATTER PLOT ────────────────────────────────────────────────
# Visual inspection — do natural groups emerge?
plt.figure(figsize=(8, 6))
plt.scatter(scores_final[:, 0], scores_final[:, 1],
            s=80, alpha=0.8, edgecolors="black", linewidths=0.5, color=COLORS.get("blue1"))

# Label each point with participant ID if available
if "study_id" in df.columns:
    for i, pid in enumerate(df["study_id"]):
        plt.annotate(pid,
                     (scores_final[i, 0], scores_final[i, 1]),
                     fontsize=7, alpha=0.7,
                     xytext=(4, 4), textcoords="offset points")

plt.xlabel(f"PC1 ({explained_var[0]:.1f}% variance)")
plt.ylabel(f"PC2 ({explained_var[1]:.1f}% variance)")
plt.title("PCA Score Plot — PC1 vs PC2")
plt.axhline(0, color="gray", linestyle="--", alpha=0.4)
plt.axvline(0, color="gray", linestyle="--", alpha=0.4)
plt.tight_layout()
date = datetime.now().strftime("%Y-%m-%d")
plt.savefig(f"../../plots/Prediction_Model/PCA/hr_pca_scatter_filtered_{date}.png", dpi=300)
plt.show()